# LoRaS-CT — With KD vs Without KD (3 datasets, single seed)

Standalone notebook. Scope: **LoRaS-CT only**, trained two ways per dataset —
**With KD** (distilled from ImageNet-pretrained ViT/DeiT/Swin teachers, logits
simply averaged, teachers NOT fine-tuned on the target dataset) and
**Without KD** (plain cross-entropy training on ground-truth labels only, no
teachers) — across **three datasets** (Kather5k, CRC7k, BreakHis).

Single seed (42), no multi-seed statistics. Both LoRaS-CT variants are
checkpointed per dataset, so an interrupted run just resumes. Output is
Accuracy (%) only — With KD vs Without KD, per dataset.

## 1. Setup

In [ ]:
!pip install --upgrade pip setuptools wheel --quiet
!pip install numpy pandas matplotlib torch torchvision timm scikit-learn thop tqdm --quiet


In [ ]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import torch
import torch.multiprocessing as mp
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass  # already set (e.g. re-running this cell) -- harmless
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split, Subset
from timm import create_model
from timm.layers import DropPath
from thop import profile
from tqdm.auto import tqdm

# ============================================================
# GLOBAL CONFIG
# ============================================================
GLOBAL_NUM_LAYERS = 2
GLOBAL_NUM_HEADS = 8
GLOBAL_RANK = 32
GLOBAL_NUM_EPOCHS = 10           # LoRaS-CT student training epochs (both KD and No-KD)

assert GLOBAL_RANK % GLOBAL_NUM_HEADS == 0, "GLOBAL_RANK must be divisible by GLOBAL_NUM_HEADS"

# Set True to ignore any saved checkpoints and force everything to retrain from scratch.
FORCE_RETRAIN = False

# Single seed -- no multi-seed statistics needed for this comparison.
SEED = 42

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)

# ============================================================
# QUICK TEST MODE -- smoke-test the whole pipeline in a few minutes before
# committing to the full run. Tiny data subsets + fewer epochs.
# ============================================================
QUICK_TEST_MODE = False
QUICK_TEST_MAX_TRAIN = 64
QUICK_TEST_MAX_VAL = 16
QUICK_TEST_MAX_TEST = 16
RUN_EPOCHS = 2 if QUICK_TEST_MODE else GLOBAL_NUM_EPOCHS

def quick_subset(torch_dataset, max_n):
    if not QUICK_TEST_MODE:
        return torch_dataset
    n = min(max_n, len(torch_dataset))
    return Subset(torch_dataset, list(range(n)))

if QUICK_TEST_MODE:
    print("QUICK_TEST_MODE is ON -- using tiny data subsets / fewer epochs to smoke-test the pipeline.")


## 2. Dataset registry

In [ ]:
# ============================================================
# DATASET REGISTRY -- replace the placeholder paths below with your real
# dataset locations.
# ============================================================
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
DATA_ROOT = "/kaggle/input" if IS_KAGGLE else os.environ.get("DATA_ROOT", "./data")
OUTPUT_ROOT = "/kaggle/working" if IS_KAGGLE else os.environ.get("OUTPUT_ROOT", "./outputs")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

DATASET_CONFIGS = {
    "Kather5k": {
        "path": "/scratch/home/admin/Kather_texture_2016_image_tiles_5000",   # <-- edit if needed
    },
    "CRC7k": {
        "path": "/scratch/home/admin/CRC-VAL-HE-7K",   # <-- edit if needed
    },
    "BreakHis": {
        "path": "/scratch/home/admin/BreaKHis_v1/BreaKHis_v1/histology_slides/breast_cache/",   # <-- edit if needed
    },
}

def _check_dataset_paths(cfg):
    paths = [cfg["path"]] if "path" in cfg else [cfg["train_val_path"], cfg["test_path"]]
    return [p for p in paths if not os.path.isdir(p)]

# ============================================================
# DATASET SELECTOR
#   - one name from DATASET_CONFIGS  -> runs the pipeline for just that dataset
#   - "ALL"                          -> runs every dataset, back-to-back
# ============================================================
SELECTED_DATASET = "ALL"   # <-- "Kather5k" | "CRC7k" | "BreakHis" | "ALL"

assert SELECTED_DATASET == "ALL" or SELECTED_DATASET in DATASET_CONFIGS, (
    f"Unknown SELECTED_DATASET '{SELECTED_DATASET}'. Choose one of {list(DATASET_CONFIGS)} or 'ALL'."
)
DATASETS_TO_RUN = list(DATASET_CONFIGS.keys()) if SELECTED_DATASET == "ALL" else [SELECTED_DATASET]
print(f"Will run the pipeline for: {DATASETS_TO_RUN}")


## 3. Patient-level leakage-fix helpers

In [ ]:
# ============================================================
# Patient/slide-level leakage-fix -- BreakHis has multiple patches per patient;
# this ensures no patient's patches appear in more than one split. Datasets
# with no parseable patient ID (Kather5k, CRC7k) fall back to the original
# image-level random split automatically.
# ============================================================
import re
from sklearn.model_selection import GroupShuffleSplit

def try_extract_patient_id(filename):
    patterns = [
        r'SOB_[A-Z]_[A-Z]+-(\d+-\d+)',   # BreakHis: SOB_B_TA-14-4659-40-001.png -> "14-4659"
        r'^(P\d+)',
        r'patient[_-]?(\d+)',
        r'case[_-]?(\d+)',
    ]
    for pat in patterns:
        m = re.search(pat, filename, flags=re.IGNORECASE)
        if m:
            return m.group(1)
    return None


def get_patient_groups(image_folder_dataset):
    filepaths = [s[0] for s in image_folder_dataset.samples]
    patient_ids = [try_extract_patient_id(os.path.basename(fp)) for fp in filepaths]
    if any(p is None for p in patient_ids) or len(set(patient_ids)) < 2:
        return None
    return np.array(patient_ids)


def patient_group_split(image_folder_dataset, seed, test_frac=0.2, val_frac_of_trainval=0.1):
    groups = get_patient_groups(image_folder_dataset)
    if groups is None:
        return None

    idx = np.arange(len(image_folder_dataset))
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(idx, groups=groups))

    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_frac_of_trainval, random_state=seed)
    tr_sub, val_sub = next(gss2.split(trainval_idx, groups=groups[trainval_idx]))
    train_idx, val_idx = trainval_idx[tr_sub], trainval_idx[val_sub]

    assert not (set(groups[train_idx]) & set(groups[test_idx]))
    assert not (set(groups[train_idx]) & set(groups[val_idx]))
    assert not (set(groups[val_idx]) & set(groups[test_idx]))
    return train_idx, val_idx, test_idx


## 4. Transforms

In [ ]:
train_val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

batch_size = QUICK_TEST_MAX_TRAIN if QUICK_TEST_MODE else 32


## 5. Model definitions (LoRaS-CT only)

In [ ]:
class ResNet18_Features(nn.Module):
    def __init__(self):
        super(ResNet18_Features, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-2])

    def forward(self, x):
        return self.features(x)  # (B, 512, 7, 7)


class DenseNet121_Features(nn.Module):
    def __init__(self):
        super(DenseNet121_Features, self).__init__()
        densenet = models.densenet121(pretrained=True)
        self.features = densenet.features

    def forward(self, x):
        x = self.features(x)
        x = F.relu(x, inplace=False)
        return x  # (B, 1024, 7, 7)


In [ ]:
class LowRankSparseMultiheadAttention(nn.Module):
    """Attention computed entirely in rank-space (dimension r) -- Q, K, V are
    never projected back up to full embed_dim before the attention product."""
    def __init__(self, embed_dim, num_heads, rank, sparsity_ratio=0.5):
        super(LowRankSparseMultiheadAttention, self).__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        assert rank % num_heads == 0, "rank must be divisible by num_heads (rank-space multi-head split)"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.rank = rank
        self.rank_head_dim = rank // num_heads
        self.sparsity_ratio = sparsity_ratio

        self.q_low = nn.Linear(embed_dim, rank, bias=False)
        self.k_low = nn.Linear(embed_dim, rank, bias=False)
        self.v_low = nn.Linear(embed_dim, rank, bias=False)
        self.rank_mix = nn.Linear(rank, rank, bias=False)
        self.out_proj = nn.Linear(rank, embed_dim, bias=False)
        self.scale = rank ** -0.5

    def sparse_attention(self, attn_scores, sparsity_ratio):
        """Masks non-top-k positions with -inf BEFORE softmax -- multiplying by 0
        and softmaxing would still give masked positions exp(0)=1, which can
        outweigh genuinely-kept positions when raw scores are negative."""
        batch_size, num_heads, seq_length, _ = attn_scores.size()
        if seq_length == 1:
            return attn_scores
        num_to_keep = max(1, int(sparsity_ratio * seq_length))
        top_scores, _ = torch.topk(attn_scores, k=num_to_keep, dim=-1)
        threshold = top_scores.min(dim=-1, keepdim=True)[0]
        sparse_mask = attn_scores >= threshold
        return attn_scores.masked_fill(~sparse_mask, float('-inf'))

    def forward(self, x):
        batch_size, seq_length, embed_dim = x.size()
        Q = self.q_low(x)
        K = self.k_low(x)
        V = self.v_low(x)
        Q = Q.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        sparse_attn_scores = self.sparse_attention(attn_scores, self.sparsity_ratio)
        attn_probs = F.softmax(sparse_attn_scores, dim=-1)
        attn_output = torch.matmul(attn_probs, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.rank)
        attn_output = self.rank_mix(attn_output)
        return self.out_proj(attn_output)


class CustomDeiTLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, rank, mlp_ratio=4., drop_path=0.1, sparsity_ratio=0.5):
        super(CustomDeiTLayer, self).__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = LowRankSparseMultiheadAttention(embed_dim, num_heads, rank, sparsity_ratio)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim), nn.GELU(), nn.Linear(mlp_hidden_dim, embed_dim),
        )

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class HybridStudentModel(nn.Module):
    """LoRaS-CT."""
    def __init__(self, num_classes, embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS,
                 rank=GLOBAL_RANK, drop_path_rate=0.1, sparsity_ratio=0.5, grid_size=3):
        super(HybridStudentModel, self).__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        self.grid_size = grid_size
        concat_channels = 512 + 1024

        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer(embed_dim, num_heads, rank, drop_path=drop_path_rate,
                             sparsity_ratio=sparsity_ratio) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        resnet_feats = self.resnet(x)
        densenet_feats = self.densenet(x)
        combined_feats = torch.cat((resnet_feats, densenet_feats), dim=1)
        if self.grid_size != combined_feats.shape[-1]:
            combined_feats = F.adaptive_avg_pool2d(combined_feats, (self.grid_size, self.grid_size))
        b, c, h, w = combined_feats.shape
        combined_feats = combined_feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(combined_feats)
        for layer in self.deit_layers:
            x = layer(x)
        x = self.norm(x)
        pooled = x.mean(dim=1)
        return self.classifier(pooled)


## 6. Shared utilities (train/eval)

In [ ]:
def evaluate_model(model, data_loader, criterion, return_predictions=False):
    model.eval()
    correct, total, test_loss = 0, 0, 0.0
    all_labels, all_preds = [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.cuda(), labels.cuda()
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            if return_predictions:
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())
    accuracy = 100 * correct / total
    avg_loss = test_loss / len(data_loader)
    if return_predictions:
        return accuracy, avg_loss, np.array(all_labels), np.array(all_preds)
    return accuracy, avg_loss


class DistillationLoss(nn.Module):
    def __init__(self, alpha=0.5, temperature=3.0):
        super(DistillationLoss, self).__init__()
        self.alpha = alpha
        self.temperature = temperature
        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_div = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, teacher_logits, ground_truth):
        hard_loss = self.ce_loss(student_logits, ground_truth)
        soft_loss = self.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=1),
            F.softmax(teacher_logits / self.temperature, dim=1)
        ) * (self.temperature ** 2)
        return self.alpha * soft_loss + (1 - self.alpha) * hard_loss


def train_model_with_distillation(student_model, teacher_models, train_loader, val_loader,
                                   distillation_criterion, optimizer, num_epochs=1, verbose=False):
    for epoch in range(num_epochs):
        student_model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"  epoch {epoch+1}/{num_epochs}", leave=False)
        for images, labels in pbar:
            images, labels = images.cuda(), labels.cuda()
            optimizer.zero_grad()
            student_outputs = student_model(images)
            with torch.no_grad():
                teacher_logits = [teacher(images) for teacher in teacher_models]
                combined_teacher_logits = sum(teacher_logits) / len(teacher_logits)
            loss = distillation_criterion(student_outputs, combined_teacher_logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        avg_loss = running_loss / len(train_loader)
        if verbose:
            val_accuracy, val_loss = evaluate_model(student_model, val_loader, distillation_criterion.ce_loss)
            print(f"  epoch [{epoch+1}/{num_epochs}] train_loss={avg_loss:.4f} "
                  f"val_loss={val_loss:.4f} val_acc={val_accuracy:.2f}%")
        else:
            print(f"  epoch [{epoch+1}/{num_epochs}] train_loss={avg_loss:.4f}")
    return student_model


def train_model_plain(model, train_loader, val_loader, criterion, optimizer, num_epochs=1, verbose=False):
    """Plain (non-distillation) training -- used for teacher fine-tuning AND
    for the 'Without KD' LoRaS-CT variant (ground-truth labels only)."""
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"  epoch {epoch+1}/{num_epochs}", leave=False)
        for images, labels in pbar:
            images, labels = images.cuda(), labels.cuda()
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        avg_loss = running_loss / len(train_loader)
        if verbose:
            val_accuracy, val_loss = evaluate_model(model, val_loader, criterion)
            print(f"  epoch [{epoch+1}/{num_epochs}] train_loss={avg_loss:.4f} "
                  f"val_loss={val_loss:.4f} val_acc={val_accuracy:.2f}%")
        else:
            print(f"  epoch [{epoch+1}/{num_epochs}] train_loss={avg_loss:.4f}")
    return model


## 7. Checkpointing

In [ ]:
# ============================================================
# Per-model checkpointing -- keyed by (dataset, model_name), covers teachers
# AND both LoRaS-CT variants. An interruption anywhere doesn't require
# retraining what already finished.
# ============================================================
def checkpoint_path(dataset_name, model_name):
    safe_name = model_name.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "-")
    return f"{OUTPUT_ROOT}/{dataset_name}_{safe_name}_checkpoint.pth"


def save_checkpoint(dataset_name, model_name, model, extra=None):
    path = checkpoint_path(dataset_name, model_name)
    payload = {'model_state_dict': model.state_dict()}
    if extra is not None:
        payload['extra'] = extra
    torch.save(payload, path)
    print(f"  [checkpoint saved] {path}")


def load_checkpoint(dataset_name, model_name, model):
    """Loads weights into `model` in-place if a checkpoint exists. Returns the
    'extra' payload (or an empty dict) on success, None if no checkpoint was
    loaded (caller should train). Respects FORCE_RETRAIN."""
    path = checkpoint_path(dataset_name, model_name)
    if FORCE_RETRAIN or not os.path.exists(path):
        return None
    ckpt = torch.load(path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"  [checkpoint found] Skipping training for '{model_name}' -- loaded from {path}")
    return ckpt.get('extra', {})


## 8. Per-dataset pipeline

In [ ]:
def load_dataset(dataset_name, dataset_cfg, seed):
    """Returns train_loader, val_loader, test_loader, num_classes."""
    is_presplit = "test_path" in dataset_cfg
    g = torch.Generator().manual_seed(seed)

    if is_presplit:
        train_val_dataset = datasets.ImageFolder(dataset_cfg["train_val_path"])
        test_dataset = datasets.ImageFolder(dataset_cfg["test_path"])
        assert train_val_dataset.classes == test_dataset.classes, (
            f"[{dataset_name}] Class mismatch between train/val and test folders: "
            f"{train_val_dataset.classes} vs {test_dataset.classes}."
        )
        num_classes = len(train_val_dataset.classes)

        val_size = int(len(train_val_dataset) * 0.1)
        train_size = len(train_val_dataset) - val_size
        train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size], generator=g)
        train_dataset.dataset.transform = train_val_transform
        val_dataset.dataset.transform = train_val_transform
        test_dataset.transform = test_transform
    else:
        dataset_path = dataset_cfg["path"]
        dataset = datasets.ImageFolder(dataset_path)
        num_classes = len(dataset.classes)

        group_split = patient_group_split(dataset, seed=seed)
        if group_split is not None:
            train_idx, val_idx, test_idx = group_split
            print(f"[{dataset_name}] Patient/slide IDs detected -- using patient-level "
                  f"GroupShuffleSplit (no patient appears in more than one split).")
            train_dataset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), train_idx)
            val_dataset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), val_idx)
            test_dataset = Subset(datasets.ImageFolder(dataset_path, transform=test_transform), test_idx)
        else:
            test_size = int(len(dataset) * 0.2)
            train_size = len(dataset) - test_size
            train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=g)
            val_size = int(train_size * 0.1)
            train_size = train_size - val_size
            train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size], generator=g)
            train_dataset.dataset.transform = train_val_transform
            val_dataset.dataset.transform = train_val_transform
            test_dataset.dataset.transform = test_transform

    train_dataset = quick_subset(train_dataset, QUICK_TEST_MAX_TRAIN)
    val_dataset = quick_subset(val_dataset, QUICK_TEST_MAX_VAL)
    test_dataset = quick_subset(test_dataset, QUICK_TEST_MAX_TEST)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, persistent_workers=True)
    return train_loader, val_loader, test_loader, num_classes


In [ ]:
def get_pretrained_teachers(num_classes):
    """ViT/DeiT/Swin, ImageNet-pretrained backbone -- NOT fine-tuned on the
    target dataset (no per-dataset training, no checkpointing needed). The
    'With KD' distillation below simply averages these three teachers' logits
    (no fine-tuning, no per-teacher weighting)."""
    teacher_vit = create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_deit = create_model('deit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_swin = create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=num_classes).cuda()
    teachers = [teacher_vit, teacher_deit, teacher_swin]
    for t in teachers:
        t.eval()
    return teachers


In [ ]:
def train_and_evaluate_lorasct(dataset_name, use_kd, teacher_models, num_classes,
                                train_loader, val_loader, test_loader):
    """Trains (or loads checkpoint for) LoRaS-CT either With KD (distilled from
    teacher_models) or Without KD (plain cross-entropy on ground-truth labels),
    then returns test Accuracy (%) only."""
    checkpoint_key = "LoRaS-CT_WithKD" if use_kd else "LoRaS-CT_WithoutKD"
    ce_criterion = nn.CrossEntropyLoss()

    set_all_seeds(SEED)
    model = HybridStudentModel(num_classes, grid_size=3).cuda()

    cached = load_checkpoint(dataset_name, checkpoint_key, model)
    if cached is None:
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
        if use_kd:
            criterion = DistillationLoss(alpha=0.5, temperature=3.0)
            model = train_model_with_distillation(model, teacher_models, train_loader, val_loader,
                                                   criterion, optimizer, num_epochs=RUN_EPOCHS)
        else:
            model = train_model_plain(model, train_loader, val_loader, ce_criterion, optimizer,
                                       num_epochs=RUN_EPOCHS)
        acc, _ = evaluate_model(model, test_loader, ce_criterion)
        print(f"  [{checkpoint_key}] Test Accuracy: {acc:.2f}%")
        save_checkpoint(dataset_name, checkpoint_key, model, extra={'Accuracy (%)': acc})
    else:
        if 'Accuracy (%)' in cached:
            acc = cached['Accuracy (%)']
        else:
            acc, _ = evaluate_model(model, test_loader, ce_criterion)

    return acc


In [ ]:
def run_dataset_pipeline(dataset_name, dataset_cfg):
    print(f"\n{'#'*80}\n# DATASET: {dataset_name}\n{'#'*80}")

    train_loader, val_loader, test_loader, num_classes = \
        load_dataset(dataset_name, dataset_cfg, seed=SEED)
    print(f"[{dataset_name}] Classes: {num_classes}")

    # Teachers: ImageNet-pretrained, NOT fine-tuned -- used only for the
    # With-KD variant, logits simply averaged (see train_model_with_distillation).
    teachers = get_pretrained_teachers(num_classes)

    print(f"\n--- LoRaS-CT | With KD ---")
    acc_with_kd = train_and_evaluate_lorasct(dataset_name, True, teachers, num_classes,
                                              train_loader, val_loader, test_loader)

    print(f"\n--- LoRaS-CT | Without KD ---")
    acc_without_kd = train_and_evaluate_lorasct(dataset_name, False, None, num_classes,
                                                 train_loader, val_loader, test_loader)

    del teachers
    torch.cuda.empty_cache()
    gc.collect()

    return {'With KD': acc_with_kd, 'Without KD': acc_without_kd}


## 9. Run all selected datasets

In [ ]:
all_results = {}

for dataset_name in DATASETS_TO_RUN:
    dataset_cfg = DATASET_CONFIGS[dataset_name]
    missing = _check_dataset_paths(dataset_cfg)
    if missing:
        print(f"[{dataset_name}] SKIPPED -- path(s) not found: {missing}")
        continue
    try:
        all_results[dataset_name] = run_dataset_pipeline(dataset_name, dataset_cfg)
    except Exception as e:
        print(f"[{dataset_name}] PIPELINE FAILED -- {type(e).__name__}: {e}")


## 10. Final results table — Accuracy (%), With KD vs Without KD

In [ ]:
results_df = pd.DataFrame(all_results).T
results_df = results_df[['With KD', 'Without KD']]
results_df['KD Gain (pp)'] = results_df['With KD'] - results_df['Without KD']
results_df = results_df.round(2)
results_df
